# Transformers in Vision
Today we will cover:
- Theory: ViT, attention mechanisms
- Exercise: Train ViT on CIFAR-10
- Mini Project: Image classifier with ViT

## Theory: Vision Transformers (ViT) & Attention

### Attention Mechanism
- Computes relationships between elements in a sequence.
- Key components: Query, Key, Value.
- Output: weighted sum of values based on similarity of queries and keys.
- Benefits: captures long-range dependencies.

### Vision Transformer (ViT)
- Splits image into patches (e.g., 16x16).
- Each patch is embedded and treated like a token.
- Transformer encoder processes sequence of patch embeddings.
- Classification token summarizes image features.
- Key Idea: Apply transformer architecture directly to images.

## Setup: CIFAR-10 Dataset Loader

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=32, shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = DataLoader(testset, batch_size=32, shuffle=False)

classes = trainset.classes
print("Classes:", classes)

## Exercise: Train ViT on CIFAR-10

In [ ]:
import timm  # PyTorch Image Models library
import torch.nn as nn
import torch.optim as optim

# Load pretrained ViT model
model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=10)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Training loop (simplified)
for epoch in range(1, 3):
    running_loss = 0.0
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch}, Loss: {running_loss/len(trainloader):.4f}")

## Mini Project: Image Classifier with ViT

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Evaluate on test set
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy on CIFAR-10 test set: {100 * correct / total:.2f}%")

# Predict on a batch of images
dataiter = iter(testloader)
images, labels = next(dataiter)

def imshow(img):
    img = img / 2 + 0.5
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

imshow(torchvision.utils.make_grid(images[:4]))
outputs = model(images.to(device))
_, predicted = torch.max(outputs, 1)
print('Predicted:', ' '.join(f'{classes[predicted[j]]}' for j in range(4)))